# Ensemble (moyenne de modèles)

Relit les prédictions sauvegardées par chaque notebook (`data/predictions/<modele>_<cible>.csv`, écrites par `trois_tests`) et fait la moyenne des modèles choisis. Pas de nouvel entraînement : il faut d'abord avoir exécuté les notebooks des modèles.

In [ ]:
import sys; sys.path.append("..")
from itertools import combinations
import pandas as pd
from evaluation import DOSSIER, resume

CIBLE = "debit"  # ou "occupation"
NOMS = ["lightgbm", "xgboost", "chronos", "timestral"]
cle = ["test", "t0", "ligne"]
P = None
for nom in NOMS:
    f = DOSSIER / f"{nom}_{CIBLE}.csv"
    if f.exists():
        d = pd.read_csv(f).rename(columns={"pred": nom})
        P = d if P is None else P.merge(d[cle + [nom]], on=cle)  # mêmes heures notées pour tous
NOMS = [n for n in NOMS if n in P]
print("modèles trouvés :", NOMS, "|", len(P), "heures notées")

In [ ]:
# Chaque modèle seul, sur les 3 tests
resume(P, cols=NOMS + ["naif_lag7j"])

In [ ]:
# Toutes les moyennes possibles de 1 à 4 modèles : score moyen des 3 arcs, pour chaque test, puis moyenne des tests
def score(p, d):  # RMSE par arc et par semaine, moyenne sur les semaines puis sur les arcs
    return (((p - d.y) ** 2).groupby([d.t0, d.axe]).mean() ** .5).groupby(level=1).mean().mean()

tests = [t for t in ["semaine_lucie", "test_final", "backtest"] if t in set(P.test)]
combis = {" + ".join(c): {t: score(P.loc[P.test == t, list(c)].mean(axis=1), P[P.test == t]) for t in tests}
          for k in range(1, len(NOMS) + 1) for c in combinations(NOMS, k)}
C = pd.DataFrame(combis).T
print(C.assign(moyenne=C.mean(axis=1)).sort_values("moyenne").round(1).to_string())

In [ ]:
# Le meilleur mélange (à choisir dans le classement ci-dessus) sur les 3 tests
MELANGE = ["lightgbm", "xgboost", "timestral"]  # <- à changer à la main (meilleur sur semaine_lucie + test_final)
P["ensemble"] = P[MELANGE].mean(axis=1)
resume(P, cols=["ensemble", *MELANGE, "naif_lag7j"])